# Balthazar Tunnel — local demo

Every `blt.*` call in this notebook executes inside a **live Balthazar flow run on the
Runner**, while the notebook itself runs in your local kernel — so you keep inline plots,
`%debug`, and variable inspection.

**Before running:** start `flows/tunnel_server.py` as a flow in Balthazar and leave it
running. It writes credentials to `~/.balthazar_tunnel.json`, which the local
`balthazar.py` shim reads.

Run the cells in order. Each one prints its result and renders its figures inline, *and*
writes a flow run back into Balthazar.

Each run also carries **the source of the cell that created it**, logged into the run,
so a run in Balthazar shows the code that produced it. That needs the run to be current
— a bare history entry has no log stream — so a cell-sourced run always goes through the
tunnel's `enter_new_flow_run` path. Set `blt.log_cell_source = False` to stop sending it.

## 0. Connect

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

import balthazar as blt  # resolves to the local tunnel shim, not the Runner's module

info = blt.ping()
print(f"connected to flow {info['flow_name']!r}")
print(f"tunnel flow run   {info['flow_run_id']}")

## 1. Read devices out of the space

`search_devices` runs on the Runner; `device.params` is fetched lazily and cached.

In [ ]:
devices = blt.search_devices()
print(f"{len(devices)} device(s) in the space\n")

print(f"The first 3 devices are\n {devices[0:5]}")

In [ ]:
devices = blt.search_devices()
print(f"{len(devices)} device(s) in the space\n")

print(f"{'name':<18}{'type':<12}params")
print("-" * 72)
for d in devices:
    print(f"{d.name:<18}{d.type:<12}{dict(d.params)}")

device = devices[0]
device_params = dict(device.params)
baseline_yield = float(device_params.get("yield_pct", 50.0))
print(f"\nusing {device.name} (baseline yield {baseline_yield}%)")

## 2. A plain run — inputs and outputs, no plot

`parameters` are the run's **inputs** and must be flat scalars (the platform infers each
type, so a `None` has no inferable type). `output` takes **primitives only**: str / int /
float / bool / small list — no dicts, no dates, no DataFrames.

In [ ]:
inputs = {
    "operator": "dejan",
    "source": "vscode-notebook",
    "baseline_yield_pct": baseline_yield,
    "dry_run": False,
}
outputs = {
    "device_name": device.name,
    "baseline_yield_pct": baseline_yield,
    "status": "success",
}

run_id = blt.new_flow_run(
    "tunnel: plain run", devices=[device], parameters=inputs, output=outputs
)

print(f"run {run_id}")
for label, payload in (("inputs", inputs), ("outputs", outputs)):
    print(f"\n{label}:")
    for k, v in payload.items():
        print(f"  {k:<22}{v!r}")

## 3. A run carrying one plot

The figure is built locally from real device data, sent to Balthazar as SVG, **and**
rendered inline below. The inline backend displays figures at the end of the cell, which
is after `new_flow_run` has already shipped them — so both happen from one `fig`.

In [ ]:
bias_min, bias_max, n_points = -1.0, 1.0, 201
bias = np.linspace(bias_min, bias_max, n_points)
current = np.sinh(bias * 4.0) * baseline_yield / 100.0

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(bias, current, color="#2563eb")
ax.axhline(0, color="0.7", lw=0.8)
ax.axvline(0, color="0.7", lw=0.8)
ax.set_xlabel("Bias (V)")
ax.set_ylabel("Current (mA)")
ax.set_title(f"I-V sweep - {device.name}")
fig.tight_layout()

resistance = float(np.gradient(bias, current)[n_points // 2])
outputs = {
    "i_at_1v_ma": round(float(current[-1]), 4),
    "i_at_minus_1v_ma": round(float(current[0]), 4),
    "zero_bias_resistance_ohm": round(resistance, 4),
    "symmetric": bool(abs(current[0] + current[-1]) < 1e-9),
    "status": "success",
}

run_id = blt.new_flow_run(
    "tunnel: I-V sweep with plot",
    devices=[device],
    parameters={
        "bias_min_v": bias_min,
        "bias_max_v": bias_max,
        "points": n_points,
        "sweep_mode": "linear",
    },
    output=outputs,
    figures=fig,
)

print(f"run {run_id} (1 plot)")
for k, v in outputs.items():
    print(f"  {k:<26}{v!r}")

## 4. A run carrying several plots

Both figures are built in **this** cell on purpose. The inline backend closes figures at
the end of each cell (`InlineBackend.close_figures` defaults to `True`), so a figure from
an earlier cell is already gone — `figures="all"` would find nothing. In a plain script
that is not an issue; in a notebook, build the set you want to ship together.

In [ ]:
rng = np.random.default_rng(0)
yields = rng.normal(baseline_yield, 4.0, 500)

fig_hist, ax1 = plt.subplots(figsize=(6, 4))
ax1.hist(yields, bins=30, color="#7c3aed")
ax1.axvline(float(yields.mean()), color="#1f2937", ls="--", lw=1.2, label="mean")
ax1.set_xlabel("Yield (%)")
ax1.set_ylabel("Count")
ax1.set_title("Yield distribution")
ax1.legend()
fig_hist.tight_layout()

fig_cdf, ax2 = plt.subplots(figsize=(6, 4))
ax2.plot(np.sort(yields), np.linspace(0, 1, yields.size), color="#059669")
ax2.set_xlabel("Yield (%)")
ax2.set_ylabel("Cumulative fraction")
ax2.set_title("Yield CDF")
fig_cdf.tight_layout()

outputs = {
    "yield_mean_pct": round(float(yields.mean()), 3),
    "yield_std_pct": round(float(yields.std()), 3),
    "yield_min_pct": round(float(yields.min()), 3),
    "yield_max_pct": round(float(yields.max()), 3),
    "figure_count": 2,
    "status": "success",
}

run_id = blt.new_flow_run(
    "tunnel: batch summary with 2 plots",
    devices=[device],
    parameters={"sample_count": int(yields.size), "rng_seed": 0, "distribution": "normal"},
    output=outputs,
    figures=[fig_hist, fig_cdf],  # or figures="all" for every figure open right now
)

print(f"run {run_id} (2 plots)")
for k, v in outputs.items():
    print(f"  {k:<20}{v!r}")

## 5. A run marked FAILED

A run is `FINISHED` unless you pass `status="FAILED"` — failures inside the run's context
are caught and reported rather than silently reddening it. This cell creates a red run on
purpose so you can see the difference in the UI.

In [ ]:
fig_fail, ax = plt.subplots(figsize=(6, 3))
ax.plot([0, 1.5, 3.2], [0, 2.1, 10.0], color="#dc2626", marker="o")
ax.axhline(10.0, color="0.6", ls="--", lw=1, label="compliance 10 mA")
ax.set_xlabel("Bias (V)")
ax.set_ylabel("Current (mA)")
ax.set_title("Aborted sweep")
ax.legend()
fig_fail.tight_layout()

run_id = blt.new_flow_run(
    "tunnel: FAILED on purpose (demo)",
    devices=[device],
    parameters={"bias_max_v": 5.0, "compliance_ma": 10.0},
    output={"aborted_at_v": 3.2, "status": "failed"},
    status="FAILED",
    error_message="sweep aborted: compliance limit hit at 3.2 V",
    figures=fig_fail,
)
print(f"run {run_id} (FAILED on purpose)")

## 6. Done

Open the tunnel flow in Balthazar: the runs above appear as its children, each with its
inputs, outputs and plots. Four runs finished green; the last one is red by design.

In [ ]:
blt.info("notebook demo finished from the local kernel")
print("check the tunnel flow's run list in Balthazar")